## Setup for LangChain with Gemini LLM

To use the Gemini API with LangChain, you'll need an API key. If you don't already have one, create a key in [Google AI Studio](https://aistudio.google.com/app/apikey).

In Colab, add the key to the secrets manager under the "🔑" icon in the left panel. Give it the name `GOOGLE_API_KEY`. This ensures your API key is stored securely and is not exposed in your notebook code.

In [36]:
# Install necessary packages
!pip uninstall -y chromadb # Uninstall chromadb to ensure a clean install via langchain-chroma
!pip install -q -U langchain-google-genai langchain langchain-chroma langchain-text-splitters langchain-community langchain-core

Found existing installation: chromadb 1.5.9
Uninstalling chromadb-1.5.9:
  Successfully uninstalled chromadb-1.5.9


Next, we'll import the required libraries and configure the API key. The `langchain_google_genai` package provides the integration with Google's Generative AI models.

In [37]:
# Import the Python SDK for secure key handling and LangChain components
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

# Get the API key from Colab secrets
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

# Initialize the Generative Model with a generally available model from the list
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=GOOGLE_API_KEY)

Now, let's create a simple prompt and invoke the LLM to get a response. We'll use a `ChatPromptTemplate` to structure our input to the model.

### Ingestion and Retrieval Workflow with LangChain, Chroma, and Gemini

This section demonstrates how to ingest an HR policy document into a Chroma vector store and then use a Gemini LLM to retrieve information and answer questions based on the document.

In [38]:
# Install the chromadb client
!pip install -q chromadb
%pip install -U chromadb tiktoken

For demonstration purposes, let's create a sample HR policy document as a string. In a real-world scenario, you would load this from a file (e.g., PDF, TXT, DOCX).

In [39]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Sample HR Policy Document (replace with your actual document content)
hr_policy_text = """
# HR Policy Document

## Section 1: Employee Code of Conduct

All employees are expected to maintain a professional and respectful work environment. This includes, but is not limited to, treating colleagues and clients with dignity, avoiding harassment, and adhering to company values. Any violations of the code of conduct will be subject to disciplinary action, up to and including termination.

### 1.1 Attendance
Employees are expected to be punctual and maintain regular attendance. Absences must be reported to a supervisor at least 24 hours in advance, except in emergencies. Excessive absenteeism may lead to disciplinary action.

### 1.2 Dress Code
A business casual dress code is enforced in the office. Specific guidelines for client-facing roles will be provided by department managers.

## Section 2: Leave Policy

### 2.1 Annual Leave
Full-time employees are eligible for 20 days of annual leave per year, accrued monthly. Leave requests must be submitted through the HR portal at least two weeks in advance and are subject to management approval.

### 2.2 Sick Leave
Employees are granted 10 days of sick leave per year. For absences exceeding three consecutive days, a doctor's note may be required.

### 2.3 Parental Leave
New parents are eligible for up to 12 weeks of paid parental leave. Details and eligibility criteria are available on the HR portal.

## Section 3: Expense Reimbursement

Employees can claim reimbursement for business-related expenses. All claims must be accompanied by original receipts and submitted within 30 days of the expense date. Expenses must be pre-approved by a department head if they exceed $500.

## Section 4: Performance Reviews

Performance reviews are conducted annually for all employees. These reviews provide an opportunity to discuss achievements, set goals, and identify areas for development. Performance is evaluated based on job responsibilities, contribution to team goals, and adherence to company values.

## Section 5: Data Privacy

Employees must adhere to strict data privacy guidelines. Confidential company and client information must not be shared with unauthorized individuals. All electronic data must be stored on approved company systems and secured with strong passwords. Violations of data privacy policy will result in immediate termination and potential legal action.
"""

# Create a Document object
document = Document(page_content=hr_policy_text, metadata={"source": "HR Policy Document"})

# Split the document into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents([document])

print(f"Original document has {len(hr_policy_text)} characters.")
print(f"Split into {len(chunks)} chunks.")
# print(chunks[0].page_content)

Original document has 2352 characters.
Split into 4 chunks.


In [40]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

# Initialize Google Generative AI Embeddings
# The GOOGLE_API_KEY is already defined from a previous cell (e587ce5d)
# Using 'models/gemini-embedding-001' which was identified as an available embedding model
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=GOOGLE_API_KEY)

# Create a Chroma vector store from the document chunks and embeddings
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings)

print("Vector store created and populated with HR policy document chunks.")

Vector store created and populated with HR policy document chunks.


Let's list the available models that support embedding to find a suitable one, as `text-embedding-004` was also not found.

In [41]:
import google.generativeai as genai

genai.configure(api_key=GOOGLE_API_KEY)

print("Available embedding models:")
for m in genai.list_models():
  if "embedContent" in m.supported_generation_methods:
    print(m.name)

Available embedding models:
models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


Now that the HR policy document is ingested into the vector store, we can set up a retrieval chain to answer questions based on the document. We will use the `gemini-2.5-flash` model as the LLM, which was initialized as `llm` in a previous cell.

In [42]:
# The original %pip install command is removed to avoid dependency conflicts and is not needed for a functional RAG setup.
# The original imports for chain functions are removed as per the request to write code without chains.
# from langchain_classic.chains.combine_documents import create_stuff_documents_chain
# from langchain_classic.chains import create_retrieval_chain

from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate

# Re-initialize llm and retriever as they were not found in the current scope.
# This assumes GOOGLE_API_KEY and chunks are available from previous successful cell executions.
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=GOOGLE_API_KEY)

# Re-initialize embeddings and retriever
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2", google_api_key=GOOGLE_API_KEY)
vectorstore = Chroma.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever()

# Define the prompt template for our LLM
# The context will be filled by the retrieved documents
prompt = ChatPromptTemplate.from_template(
    """Answer the user's question based on the provided context.
    If you don't know the answer, just say that you don't know, don't try to make up an answer.

    Context: {context}
    Question: {input}"""
)

# Define a custom class to simulate the behavior of the retrieval_chain
# without using Langchain's chain objects.
class CustomRetrievalChain:
    def __init__(self, llm_model, retriever_model):
        self.llm = llm_model
        self.retriever = retriever_model

    def invoke(self, input_dict):
        question = input_dict["input"]

        # 1. Retrieve relevant documents using the retriever
        retrieved_docs = self.retriever.invoke(question)

        # 2. Format the retrieved documents into a single context string
        #    Each document's page_content is joined by double newlines.
        formatted_context = "\n\n".join([doc.page_content for doc in retrieved_docs])

        # 3. Format the prompt with the context and the user's question
        #    ChatPromptTemplate.format_messages returns a list of Message objects.
        formatted_prompt_messages = prompt.format_messages(
            context=formatted_context,
            input=question
        )

        # 4. Invoke the LLM with the formatted prompt messages
        response = self.llm.invoke(formatted_prompt_messages)

        # The result should match the structure of the original retrieval_chain's output.
        return {"answer": response.content}

# Diagnostic prints to check variable states
print(f"[Diagnostic] GOOGLE_API_KEY available: {bool(GOOGLE_API_KEY)}")
print(f"[Diagnostic] Type of llm before instantiation: {type(llm) if 'llm' in locals() or 'llm' in globals() else 'Undefined'}")
print(f"[Diagnostic] Type of retriever before instantiation: {type(retriever) if 'retriever' in locals() or 'retriever' in globals() else 'Undefined'}")

# Instantiate the custom retrieval chain, so subsequent cells can call retrieval_chain.invoke()
retrieval_chain = CustomRetrievalChain(llm, retriever)

print("RAG 'retrieval_chain' (manual implementation) created successfully.")


[Diagnostic] GOOGLE_API_KEY available: True
[Diagnostic] Type of llm before instantiation: <class 'langchain_google_genai.chat_models.ChatGoogleGenerativeAI'>
[Diagnostic] Type of retriever before instantiation: <class 'langchain_core.vectorstores.base.VectorStoreRetriever'>
RAG 'retrieval_chain' (manual implementation) created successfully.


In [46]:
# Ask a question about the HR policy
question = "What is the policy on annual leave?"
response = retrieval_chain.invoke({"input": question})

print(f"Question: {question}\n")
print("Answer:")
from IPython.display import Markdown, display
# Extract text content from the list of parts in response['answer']
if isinstance(response['answer'], list):
    full_text_content_answer = "".join([part.get('text', '') for part in response['answer']])
else:
    full_text_content_answer = str(response['answer'])
display(Markdown(full_text_content_answer))

print("\n---\n")

question_two = "What are the rules regarding data privacy?"
response_two = retrieval_chain.invoke({"input": question_two})

print(f"Question: {question_two}\n")
print("Answer:")
if isinstance(response_two['answer'], list):
    full_text_content_answer_two = "".join([part.get('text', '') for part in response_two['answer']])
else:
    full_text_content_answer_two = str(response_two['answer'])
display(Markdown(full_text_content_answer_two))

Question: What is the policy on annual leave?

Answer:


I don't know the answer. The provided context only states "Section 2: Leave Policy" but does not contain any details regarding annual leave.


---

Question: What are the rules regarding data privacy?

Answer:


Based on the provided context, the rules regarding data privacy are:

*   Employees must adhere to strict data privacy guidelines.
*   Confidential company and client information must not be shared with unauthorized individuals.
*   All electronic data must be stored on approved company systems and secured with strong passwords.
*   Violations of data privacy policy will result in immediate termination and potential legal action.